# DuckDB SQL

In [1]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
minio_user = os.getenv("MINIO_ROOT_USER", "minioadmin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS (
        TYPE S3,
        KEY_ID '{minio_user}',
        SECRET '{minio_password}',
        ENDPOINT 'localhost:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")
print("Connected to DuckDB and MinIO successfully")

Connected to DuckDB and MinIO successfully


# Basic sqls

In [2]:
# query ot count total no of rows in whole file
q1 = """
SELECT COUNT(*) as total_rows from read_parquet('s3://analytics-data/ml_features.parquet')"""
res1 = con.execute(q1).df()
display(res1)

,total_rows
0,515268


In [3]:
# query to describe the cols in the file 
q2 = """ DESCRIBE SELECT * FROM read_parquet('s3://analytics-data/ml_features.parquet')  """
res2 = con.execute(q2).df()
display(res2)

,column_name,column_type,null,key,default,extra
0,asset_symbol,VARCHAR,YES,None,None,None
1,asset_class,VARCHAR,YES,None,None,None
2,exchange,VARCHAR,YES,None,None,None
3,interval,VARCHAR,YES,None,None,None
4,date,TIMESTAMP,YES,None,None,None
5,open,DOUBLE,YES,None,None,None
6,high,DOUBLE,YES,None,None,None
7,low,DOUBLE,YES,None,None,None
8,close,DOUBLE,YES,None,None,None
9,volume,DOUBLE,YES,None,None,None


In [4]:
# query to read parquet from ml_features where to read specific asset on specific interval 
q3 = """
SELECT date, asset_symbol, open, high, low, close, volume
FROM read_parquet('s3://analytics-data/ml_features.parquet')
WHERE asset_symbol = 'AAPL' AND interval = '1h'
ORDER BY date ASC
LIMIT 5;
"""
res3 = con.execute(q3).df()
display(res3)

,date,asset_symbol,open,high,low,close,volume
0,2024-05-13 16:30:00,AAPL,186.289993,186.929993,186.250000,186.658600,8098301.0
1,2024-05-13 17:30:00,AAPL,186.660004,186.949997,186.369995,186.650101,5453676.0
2,2024-05-13 18:30:00,AAPL,186.659897,187.100006,186.559998,186.989899,6516486.0
3,2024-05-13 19:30:00,AAPL,186.985001,187.029999,186.070007,186.285004,9250536.0
4,2024-05-14 13:30:00,AAPL,187.649994,188.300003,186.550003,187.009995,14754549.0


In [5]:
#query to get total no of rows in asset aapl for specific interval 
q4 = """
SELECT count(*) as total_rows 
FROM read_parquet('s3://analytics-data/ml_features.parquet') 
WHERE asset_symbol = 'AAPL'AND interval = '1h';
"""
res4 = con.execute(q4).df()
display(res4)

,total_rows
0,3193


# Advanced Sqls

# Moving AVG using window functions

In [6]:
# query for 7 day moving avg 

q5 = """ 
SELECT date,asset_symbol,close,
AVG(close) OVER (
PARTITION BY asset_symbol 
ORDER BY date asc
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
) AS mv_avg_7_days
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""

res5 = con.execute(q5).df()
display(res5)


,date,asset_symbol,close,mv_avg_7_days
0,2026-03-13,AAPL,250.690002,257.919998
1,2026-03-12,AAPL,255.759995,259.609996
2,2026-03-11,AAPL,260.809998,260.751426
3,2026-03-10,AAPL,261.109985,261.309998
4,2026-03-09,AAPL,259.880005,261.748570
5,2026-03-06,AAPL,256.899994,263.615714
6,2026-03-05,AAPL,260.290009,266.091431
7,2026-03-04,AAPL,262.519989,267.784289
8,2026-03-03,AAPL,263.750000,268.307146
9,2026-03-02,AAPL,264.720001,268.425716


In [ ]:
# today close and yesterday close  

q6 = """
select date,asset_symbol, close as todays_close,
lag(close,1) over (partition by asset_symbol order by date asc) as yesterday_close
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""
res6 = con.execute(q6).df()
display(res6)

,date,asset_symbol,todays_close,yesterday_close
0,2026-03-13,AAPL,250.690002,255.759995
1,2026-03-12,AAPL,255.759995,260.809998
2,2026-03-11,AAPL,260.809998,261.109985
3,2026-03-10,AAPL,261.109985,259.880005
4,2026-03-09,AAPL,259.880005,256.899994
5,2026-03-06,AAPL,256.899994,260.290009
6,2026-03-05,AAPL,260.290009,262.519989
7,2026-03-04,AAPL,262.519989,263.750000
8,2026-03-03,AAPL,263.750000,264.720001
9,2026-03-02,AAPL,264.720001,264.179993


In [10]:
# query for day over day percentage change 
q7 = """
select date,asset_symbol, close as todays_close,
lag(close,1) over (partition by asset_symbol order by date asc) as yesterday_close,
((close - LAG(close, 1) OVER (PARTITION BY asset_symbol ORDER BY date ASC)) / LAG(close, 1) OVER (PARTITION BY asset_symbol ORDER BY date ASC)) * 100 AS daily_percent_change
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""

res7 = con.execute(q7).df()
display(res7)



,date,asset_symbol,todays_close,yesterday_close,daily_percent_change
0,2026-03-13,AAPL,250.690002,255.759995,-1.982324
1,2026-03-12,AAPL,255.759995,260.809998,-1.936277
2,2026-03-11,AAPL,260.809998,261.109985,-0.114889
3,2026-03-10,AAPL,261.109985,259.880005,0.473288
4,2026-03-09,AAPL,259.880005,256.899994,1.159989
5,2026-03-06,AAPL,256.899994,260.290009,-1.302399
6,2026-03-05,AAPL,260.290009,262.519989,-0.849452
7,2026-03-04,AAPL,262.519989,263.750000,-0.466355
8,2026-03-03,AAPL,263.750000,264.720001,-0.366425
9,2026-03-02,AAPL,264.720001,264.179993,0.204409


# Group BY ROLLUP, GROUPING SETS, CUBE

In [ ]:
# rollup 
q8 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by rollup (asset_symbol,interval)
order by asset_symbol, interval
"""

res8 = con.execute(q8).df()
display(res8)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,1d,4.241144e+13
1,1000PEPE,1h,7.834460e+13
2,1000PEPE,None,1.207560e+14
3,AAPL,1d,5.077818e+11
4,AAPL,1h,1.779542e+10
...,...,...,...
63,XRP,1d,7.191028e+11
64,XRP,1h,7.651137e+11
65,XRP,W,1.525064e+11
66,XRP,None,1.636723e+12


In [ ]:
# grouing sets 
q9 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by grouping sets (asset_symbol,interval)
order by asset_symbol, interval
"""

res9 = con.execute(q9).df()
display(res9)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,None,1.207560e+14
1,AAPL,None,7.918394e+11
2,ADA,None,8.645117e+11
3,AMZN,None,4.495029e+11
4,AVAX,None,1.320505e+10
5,BTC,None,4.894912e+08
6,DOGE,None,6.801600e+12
7,DOT,None,3.236949e+10
8,DYDX,None,6.351448e+10
9,ETH,None,4.290760e+09


In [ ]:
# cube 
q10 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by cube (asset_symbol,interval)
order by asset_symbol, interval
"""

res10 = con.execute(q10).df()
display(res10)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,1d,4.241144e+13
1,1000PEPE,1h,7.834460e+13
2,1000PEPE,None,1.207560e+14
3,AAPL,1d,5.077818e+11
4,AAPL,1h,1.779542e+10
...,...,...,...
67,None,1d,4.791163e+13
68,None,1h,8.261162e+13
69,None,1wk,9.687836e+11
70,None,W,1.232652e+12


# Ranking 